In [1]:
import pandas as pd

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
dev = pd.read_csv('data/dev.csv')

df = pd.concat([train, test, dev], ignore_index=True)

print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df.duplicated().sum())
df.head()

(16163, 3)
text            str
label         int64
label_text      str
dtype: object
text          0
label         0
label_text    0
dtype: int64
3080


,text,label,label_text
0,I am still waiting on my card?,11,card_arrival
1,What can I do if my card still hasn't arrived ...,11,card_arrival
2,I have been waiting over a week. Is the card s...,11,card_arrival
3,Can I track my card while it is in the process...,11,card_arrival
4,"How do I know if I will get my card, or if it ...",11,card_arrival


In [2]:
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(df.shape)

(13083, 3)


In [3]:
df_sample = df.groupby('label_text').sample(n=6, random_state=42, replace=True).reset_index(drop=True)

print(df_sample.shape)
print(df_sample['label_text'].nunique())
print(df_sample.head())

(462, 3)
77
                                                text  label  \
0  how does my statement get updated so i know wh...     51   
1         Why isn't a recent refund on my statement?     51   
2                    I am still waiting for a refund     51   
3  I can't see a refund for a return I completed ...     51   
4                    what is the status of my refund     51   

              label_text  
0  Refund_not_showing_up  
1  Refund_not_showing_up  
2  Refund_not_showing_up  
3  Refund_not_showing_up  
4  Refund_not_showing_up  


In [10]:
import json
import time
import pandas as pd
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def generate_bot_response(query: str) -> str:
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a helpful banking support chatbot. Answer customer questions clearly and concisely."},
            {"role": "user", "content": query}
        ]
    )
    return response.choices[0].message.content

def evaluate_response(query: str, bot_response: str, retries: int = 2) -> dict:
    prompt = f"""
    Customer query: {query}
    Bot response: {bot_response}
    
    Evaluate the bot response. Return ONLY a JSON object, no explanation:
    {{
        "relevance_score": <float 0.0-1.0>,
        "hallucination": <true or false>,
        "escalation_needed": <true or false>
    }}
    """
    for attempt in range(retries):
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": """You are a QA evaluator for a banking chatbot. 
Evaluate ONLY the informational content of the response, ignore politeness phrases 
like 'I'd be happy to help', 'Great question', etc.
Return ONLY valid JSON, nothing else."""},
                {"role": "user", "content": prompt}
            ]
        )
        raw = response.choices[0].message.content.strip()
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            time.sleep(1)
    
    return {"relevance_score": None, "hallucination": None, "escalation_needed": None}

def run_evaluation(df: pd.DataFrame) -> pd.DataFrame:
    results = []
    for i, row in df.iterrows():
        query = row['text']
        category = row['label_text']
        
        bot_response = generate_bot_response(query)
        time.sleep(0.5)
        
        scores = evaluate_response(query, bot_response)
        time.sleep(0.5)
        
        results.append({
            "query": query,
            "category": category,
            "bot_response": bot_response,
            "relevance_score": scores.get("relevance_score"),
            "hallucination": scores.get("hallucination"),
            "escalation_needed": scores.get("escalation_needed")
        })
        
        if i % 10 == 0:
            print(f"Processed {i}/{len(df)} queries")
    
    return pd.DataFrame(results)

In [11]:
df_test = df_sample.head(5)
results = run_evaluation(df_test)
results

Processed 0/5 queries


,query,category,bot_response,relevance_score,hallucination,escalation_needed
0,how does my statement get updated so i know wh...,Refund_not_showing_up,Your bank statement is typically updated in re...,0.80,False,False
1,Why isn't a recent refund on my statement?,Refund_not_showing_up,There could be several reasons why a recent re...,0.85,False,False
2,I am still waiting for a refund,Refund_not_showing_up,I'd be happy to help you with that. Can you pl...,0.60,False,False
3,I can't see a refund for a return I completed ...,Refund_not_showing_up,I'd be happy to help you with that. Can you pl...,0.60,False,False
4,what is the status of my refund,Refund_not_showing_up,"To check the status of your refund, I'll need ...",0.80,False,False


In [14]:
results_df = run_evaluation(df_sample)
os.makedirs('data', exist_ok=True)
results_df.to_csv('data/evaluation_results.csv', index=False)
print(f"Done. {len(results_df)} rows evaluated.")
print(results_df[['relevance_score', 'hallucination', 'escalation_needed']].describe())

Processed 0/462 queries
Processed 10/462 queries
Processed 20/462 queries
Processed 30/462 queries
Processed 40/462 queries
Processed 50/462 queries
Processed 60/462 queries
Processed 70/462 queries
Processed 80/462 queries
Processed 90/462 queries
Processed 100/462 queries
Processed 110/462 queries
Processed 120/462 queries
Processed 130/462 queries
Processed 140/462 queries
Processed 150/462 queries
Processed 160/462 queries
Processed 170/462 queries
Processed 180/462 queries
Processed 190/462 queries
Processed 200/462 queries
Processed 210/462 queries
Processed 220/462 queries
Processed 230/462 queries
Processed 240/462 queries
Processed 250/462 queries
Processed 260/462 queries
Processed 270/462 queries
Processed 280/462 queries
Processed 290/462 queries
Processed 300/462 queries
Processed 310/462 queries
Processed 320/462 queries
Processed 330/462 queries
Processed 340/462 queries
Processed 350/462 queries
Processed 360/462 queries
Processed 370/462 queries
Processed 380/462 queri

In [15]:
import os
os.makedirs('data', exist_ok=True)
results_df.to_csv('data/evaluation_results.csv', index=False)

print(f"Total evaluated: {len(results_df)}")
print(f"\nRelevance score stats:")
print(results_df['relevance_score'].describe())
print(f"\nHallucination rate: {results_df['hallucination'].mean():.1%}")
print(f"Escalation rate: {results_df['escalation_needed'].mean():.1%}")
print(f"None values: {results_df['relevance_score'].isna().sum()}")

Total evaluated: 462

Relevance score stats:
count    462.000000
mean       0.783247
std        0.172156
min        0.000000
25%        0.800000
50%        0.800000
75%        0.900000
max        1.000000
Name: relevance_score, dtype: float64

Hallucination rate: 3.2%
Escalation rate: 16.7%
None values: 0


In [18]:
category_stats = results_df.groupby('category').agg(
    avg_relevance=('relevance_score', 'mean'),
    hallucination_rate=('hallucination', 'mean'),
    escalation_rate=('escalation_needed', 'mean'),
    count=('query', 'count')
).round(3).sort_values('avg_relevance')

print(category_stats.head(10))

                                                  avg_relevance  \
category                                                          
failed_transfer                                           0.567   
card_payment_wrong_exchange_rate                          0.575   
balance_not_updated_after_cheque_or_cash_deposit          0.583   
Refund_not_showing_up                                     0.625   
extra_charge_on_statement                                 0.625   
transaction_charged_twice                                 0.633   
transfer_fee_charged                                      0.650   
exchange_charge                                           0.658   
declined_cash_withdrawal                                  0.663   
wrong_exchange_rate_for_cash_withdrawal                   0.667   

                                                  hallucination_rate  \
category                                                               
failed_transfer                                    

In [19]:
category_stats.to_csv('data/category_stats.csv')
print("Saved.")

Saved.
